# Extracción de metadatos de estudios
Este notebook lee imágenes de diferentes estudios y extrae sus respectivos metadatos para exportarlos en un archivo de Excel

Elaborado por Juan Manuel Rivera/AJS

# Importación de librerías y funciones auxiliares

In [1]:
import os
import pandas as pd
import pydicom
import re
import numpy as np

En la siguiente celda está la dirección del directorio donde están las imágenes

In [2]:
path = 'D:/jm.rivera/AISD/DICOM'

In [3]:
def list_one_image_per_folder(path):
    images_path = []
    for item in os.listdir(path):
        item_path = f'{path}/{item}'
        if os.path.isdir(item_path):
            images_path += list_one_image_per_folder(item_path)
        elif os.path.isfile(item_path) and '.dcm' in item:
            images_path.append(item_path)
            break
        else:
            print(f'Error al procesar el documento {item_path}')
    return images_path

In [31]:
def list_dicom_image_folders(path):
    images_path = []
    for item in os.listdir(path):
        item_path = f'{path}/{item}'
        if os.path.isdir(item_path):
            images_path += list_dicom_image_folders(item_path)
        elif os.path.isfile(item_path) and '.dcm' in item:
            images_path.append(path)
            break
        else:
            print(f'Error al procesar el documento {item_path}')
    return images_path

# Extracción de metadatos

Si desea añadir algún metadato añada el DICOM Tag en la siguiente lista

In [54]:
tags = {
    # File path
    "file_path" : [],
    # Mean z-pacing
    "mean_spacing" : [],
    # standard deviarion of z-pacing
    "sd_spacing" : [],
    # Number of slices
    "n_slices" : [],
    # Pixel Spacing
    "00280030" : [],
    # Slice thickness
    "00180050" : [],
    # Institution name
    "00080080" : [],
    # Manufacturer
    "00080070" : [],
    # Manufacturer model
    "00081090" : [],
    # kiloVolt_peak
    "00180060" : [],
    # Exposure time
    "00181150" : [],
    # Xray tube current
    "00181151" : [],
    # Exposure mAs
    "00181152" : [],
    # Protocol name
    "00181030" : [],
    # Series description
    "0008103E" : [],
    # Patient ID
    "00100020" : [],
    # Accsesion number
    "00080050" : [],
    # Patient age
    "00101010" : [],
    # Patient sex
    "00100040" : []
}

In [48]:
dicom_folders = list_dicom_image_folders(path)
print(f'Se van a procesar {len(dicom_folders)} estudios')

Se van a procesar 397 estudios


In [55]:
for i, img_path in enumerate(dicom_folders):
    tags_to_read = True
    z_positions = []
    for file in os.listdir(img_path):
        if '.dcm' in file:
            img = pydicom.dcmread(os.path.join(img_path, file))
            if tags_to_read:
                for tag, array in tags.items():
                    if re.fullmatch("\d{8}", tag) and tag in img:
                        array.append(img[tag].value)
                    elif tag == "file_path":
                        array.append(img_path)
                    elif tag in ["mean_spacing", "sd_spacing", "n_slices"]:
                        pass
                    else:
                        array.append('')
                tags_to_read = False
            z_positions.append(img['00200032'].value[2])
        
    z_positions = sorted(z_positions)
    diffs = np.diff(z_positions)
    tags["mean_spacing"].append(np.mean(diffs))
    tags["sd_spacing"].append(np.round(np.std(diffs),3))
    tags["n_slices"].append(len(z_positions))

    assert(len(tags['file_path']) == i+1)


In [56]:
df = pd.DataFrame(data=tags)
df.rename(columns={
    "file_path" : "Dirección del estudio",
    "mean_spacing" : "Media de diatancia z-axis",
    "sd_spacing" : "Desviación estándar de diatancia z-axis",
    "n_slices" : "Número de cortes",
    "00280030" : "Pixel spacing (mm)",
    "00180050" : "Slice thickness (mm)",
    "00080080" : "Institution name",
    "00080070" : "Manufacturer",
    "00081090" : "Manufacturer model",
    "00180060" : "Peak kilo voltage output",
    "00181150" : "Exposure time (miliseconds)",
    "00181151" : "Xray tube current (mA)",
    "00181152" : "Exposure (mAs)",
    "00181030" : "Protocol name",
    "0008103E" : "Series description",
    "00100020" : "Patient ID",
    "00080050" : "Accsesion number",
    "00101010" : "Patient age",
    "00100040" : "Patient sex"
    },
    inplace=True
    )
display(df.head(5))

,Dirección del estudio,Media de diatancia z-axis,Desviación estándar de diatancia z-axis,Número de cortes,Pixel spacing (mm),Slice thickness (mm),Institution name,Manufacturer,Manufacturer model,Peak kilo voltage output,Exposure time (miliseconds),Xray tube current (mA),Exposure (mAs),Protocol name,Series description,Patient ID,Accsesion number,Patient age,Patient sex
0,D:/jm.rivera/AISD/DICOM/0019983,8.000000,0.000,17,"[0.4375, 0.4375]",8.0,0510091263d5f89a7ca555058278ce32,SIEMENS,SOMATOM Perspective,130.0,2000,135,270,001_HeadSeq,,0019983,11387042,084Y,F
1,D:/jm.rivera/AISD/DICOM/0021023,9.264133,0.212,16,"[0.4296875, 0.4296875]",9.0,010168a5488f5299f80c8ac718ab5894,SIEMENS,Emotion 6 (2007),130.0,1500,167,250,01HeadSeqAuto,,0021023,6981289,55,M
2,D:/jm.rivera/AISD/DICOM/0021061,3.004488,0.078,42,"[0.404296875, 0.404296875]",3.0,10b2a6cf0fa7bd5ccd4f29f472b8f8ce,SIEMENS,Emotion 6,130.0,1500,167,250,01HeadSeqAuto,,0021061,10979790,,M
3,D:/jm.rivera/AISD/DICOM/0021092,9.002615,0.034,14,"[0.50390625, 0.50390625]",9.0,010168a5488f5299f80c8ac718ab5894,SIEMENS,Emotion 6,130.0,1500,167,250,02HeadSeq,,0021092,8020693,75,M
4,D:/jm.rivera/AISD/DICOM/0021822,8.000000,0.000,19,"[0.4375, 0.4375]",8.0,0510091263d5f89a7ca555058278ce32,SIEMENS,SOMATOM Perspective,130.0,2000,135,270,001_HeadSeq,,0021822,11389749,079Y,M


In [59]:
df.to_excel("MetadatosAISD.xlsx")